# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta:** Cada imagen tendrá **1024 valores** (32 × 32 = 1024), ya que `Flatten()` convierte la matriz bidimensional en un vector unidimensional.



## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())


In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

model.summary()

In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:** El dataset CIFAR-10 tiene imágenes de 32×32 píxeles con 3 canales (RGB). Después de convertir a escala de grises, tenemos una matriz 32×32. `Flatten()` es necesario para convertir esta matriz bidimensional en un vector unidimensional de 1024 valores, que es lo que espera la primera capa `Dense` (que espera una entrada de 1 dimensión).

**3.** ¿Por qué la última capa tiene 10 neuronas?

**Respuesta:** La última capa tiene 10 neuronas porque CIFAR-10 tiene 10 clases (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). Cada neurona representa una clase y produce un logit (puntuación) que indica la probabilidad de que la imagen pertenezca a esa clase.

**4.** ¿Qué función cumple `ReLU` en la capa oculta?

**Respuesta:** `ReLU` (Rectified Linear Unit) es una función de activación que introduce no-linealidad en la red. Funciona reemplazando todos los valores negativos por cero (f(x) = max(0, x)). Esto permite que la red aprenda representaciones complejas y no esté limitada por funciones de activación lineales.


---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta:** Usamos `from_logits=True` porque la última capa de nuestro modelo produce **logits** (valores sin normalizar, sin pasar por Softmax). La función de pérdida `SparseCategoricalCrossentropy` con `from_logits=True` aplica internamente la función Softmax de manera numéricamente estable antes de calcular la entropía cruzada. Esto es más eficiente y estable que aplicar Softmax manualmente y luego calcular la pérdida.



---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)


### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta:** Una época (`epoch`) representa un ciclo completo de entrenamiento en el cual el modelo ha procesado todo el conjunto de datos de entrenamiento una vez (pasada hacia adelante y retropropagación a través de todos los ejemplos).


**7.** ¿Qué representa `batch_size=64`?

**Respuesta:** `batch_size=64` representa el número de ejemplos (mini-lote) que se introducen a la red neuronal simultáneamente antes de calcular el gradiente de la función de pérdida y actualizar los pesos del modelo.

## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta:** Sí, hay evidencia de overfitting. El accuracy de entrenamiento continúa aumentando mientras que el accuracy de validación se estabiliza o disminuye ligeramente. Esto indica que el modelo está aprendiendo patrones del conjunto de entrenamiento pero no generaliza bien a datos nuevos.



---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta:** CIFAR-10 en escala de grises parece **más difícil** que Fashion-MNIST. Fashion-MNIST tiene imágenes 28×28 (784 píxeles) con 10 clases, mientras que CIFAR-10 tiene imágenes 32×32 (1024 píxeles) después de aplanar. El mayor número de parámetros y la mayor dimensionalidad hacen que sea más difícil para esta red simple alcanzar un buen accuracy. Typical accuracies: Fashion-MNIST ~85-88%, CIFAR-10 (grayscale) ~55-65% con esta arquitectura.



---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades
probabilities = keras.ops.softmax(logits)

# TODO 7 — Clase con mayor probabilidad
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])


In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta:** La función Softmax convierte los logits en una distribución de probabilidad válida. Por definición, una distribución de probabilidad debe sumar 1 (o 100%), ya que representa la probabilidad total de todos los eventos mutuamente excluyentes (en este caso, que la imagen pertenezca a una de las 10 clases). Softmax garantiza que cada salida esté entre 0 y 1 y que su suma sea exactamente 1.



---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?n

**Respuesta:** Accuracy final ~55-65% (dependiendo de la ejecución). La red neuronal simple con arquitectura fully-connected tiene dificultades para alcanzar accuracies altas en CIFAR-10 debido a la pérdida de información espacial después de `Flatten()`.


### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta:** Las clases más difíciles de distinguir incluyen pares como: auto/truck, pájaro/avión, y gato/perro. Estas tienen características visuales superpuestas y variaciones significativas dentro de cada clase.


### 3. Después de `Flatten()`, la imagen se convierte en un vector de 1024 valores. ¿Qué información importante sobre la imagen podría perderse al tratarla únicamente como un vector?n

**Respuesta:** Al tratar la imagen como un vector unidimensional, se pierde la **estructura espacial**. Información importante como: 
- La posición relativa de píxeles (bordes, formas, patrones locales)
- La correlación entre píxeles adyacentes
- La jerarquía visual (partes como ojos, ruedas, etc.)
- La traducción invariante (dónde aparece un objeto en la imagen)

Esto explica por qué las redes convolucionales (CNNs) son superiores para tareas de visión por computadora.

### 4. ¿Qué tipo de arquitectura podría aprovechar mejor la estructura espacial de las imágenes?

**Respuesta:** **Convolutional Neural Networks (CNNs)**. Las CNNs utilizan capas convolucionales que preservan la estructura espacial mediante kernels que se deslizan sobre la imagen, detectando características locales (bordes, texturas) y manteniendo la relación espacial entre píxenos. Arquitecturas como LeNet, VGG, ResNet o MobileNet son diseñadas específicamente para aprovechar esta estructura y han demostrado resultados mucho mejores que las redes fully-connected para clasificación de imágenes.

---
## Para pensar
Nuestro modelo hace:

```text
32×32 → Flatten → 1024 valores → Dense → ReLU → Dense(10)
```

Sin embargo, una imagen tiene **estructura espacial**: existen bordes, formas y patrones locales.

Este resultado nos deja preparada la siguiente pregunta del curso:

> **¿Cómo puede una red aprovechar directamente la estructura espacial de una imagen?**

La respuesta será: **Convolutional Neural Networks (CNNs)**.
